<a href="https://colab.research.google.com/github/harshitha020505/DLLAB/blob/main/DLLAB10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

transform = transforms.ToTensor()

train_data = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_data = datasets.MNIST(root='./data', train=False, transform=transform)

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader = DataLoader(test_data, batch_size=128)

100%|██████████| 9.91M/9.91M [00:00<00:00, 70.9MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 34.0MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 82.8MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 2.00MB/s]


In [2]:
class UndercompleteAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 128),
            nn.ReLU(),
            nn.Linear(128, 32)
        )
        self.decoder = nn.Sequential(
            nn.Linear(32, 128),
            nn.ReLU(),
            nn.Linear(128, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(-1, 784)
        z = self.encoder(x)
        out = self.decoder(z)
        return out

In [3]:
class OvercompleteAE(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(784, 1024),
            nn.ReLU(),
            nn.Linear(1024, 900)
        )
        self.decoder = nn.Sequential(
            nn.Linear(900, 1024),
            nn.ReLU(),
            nn.Linear(1024, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        x = x.view(-1, 784)
        z = self.encoder(x)
        return self.decoder(z)

In [4]:
def train(model, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        for img, _ in train_loader:
            output = model(img)
            loss = loss_fn(output, img.view(-1, 784))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

In [5]:
class RegularizedAE(UndercompleteAE):
    pass

def train_with_l2(model, epochs=5):
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        for img, _ in train_loader:
            output = model(img)
            loss = loss_fn(output, img.view(-1, 784))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [6]:
class DenoisingAE(UndercompleteAE):
    pass

def add_noise(x):
    noise = torch.randn_like(x) * 0.5
    return torch.clamp(x + noise, 0., 1.)

def train_denoising(model):
    optimizer = optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    for epoch in range(5):
        for img, _ in train_loader:
            noisy = add_noise(img)
            output = model(noisy)
            loss = loss_fn(output, img.view(-1, 784))

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

In [7]:
from sklearn.decomposition import PCA

data = train_data.data.view(-1, 784).float().numpy()

pca = PCA(n_components=32)
reduced = pca.fit_transform(data)
reconstructed = pca.inverse_transform(reduced)

In [8]:
class SparseAE(UndercompleteAE):
    def __init__(self):
        super().__init__()
        self.rho = 0.05
        self.beta = 1e-3

    def kl_divergence(self, rho_hat):
        rho = self.rho
        return rho * torch.log(rho/rho_hat) + (1-rho)*torch.log((1-rho)/(1-rho_hat))

    def forward(self, x):
        x = x.view(-1, 784)
        z = self.encoder(x)
        self.rho_hat = torch.mean(z, dim=0)
        return self.decoder(z)

In [9]:
class ContractiveAE(UndercompleteAE):
    def contractive_loss(self, x, output, lam=1e-4):
        mse = nn.MSELoss()(output, x.view(-1, 784))
        W = self.encoder[0].weight
        h = self.encoder(x.view(-1, 784))
        contractive = lam * torch.sum(W**2)
        return mse + contractive